In [4]:
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
from gensim.models import Word2Vec
import nltk
from nltk.corpus import stopwords

# Download the stopwords list from NLTK
nltk.download('stopwords')
stop_words = set(stopwords.words('english'))


ModuleNotFoundError: No module named 'matplotlib.backends.registry'

In [168]:
df = pd.read_json('yelp_academic_dataset_review.json', lines=True ,nrows=5000)
userDf = pd.read_json('yelp_academic_dataset_business.json', lines=True ,nrows=5000)

# Merge the DataFrames on the 'business_id' column


### CHECK IF THERE ARE ANY NULL VALUES

In [ ]:
userDf.isnull().sum()

In [ ]:
userDf.isnull().sum()

In [171]:
userDf= userDf.dropna()
df= df.dropna()

In [ ]:
df.drop_duplicates(subset=['text'])
userDf.drop_duplicates(subset=['latitude'])
userDf.drop_duplicates(subset=['longitude'])
merged_df = pd.merge(df, userDf, on='business_id', how='inner')

# Display the merged DataFrame
print(merged_df.head())


In [ ]:
spatial_data = merged_df[['latitude','longitude']]
spatial_data.head()

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

def cluster_data(df_decade):
    # Step 1: Standardize the data
    scaler = StandardScaler()
    df_decade_scaled = df_decade.copy()
    df_decade_scaled.iloc[:, :] = scaler.fit_transform(df_decade)
    inertia = []
    for i in range(1, 11):
        kmeans = KMeans(n_clusters=i, random_state=42)
        kmeans.fit(df_decade_scaled)
        inertia.append(kmeans.inertia_)
    # Plot the elbow graph
    plt.figure(figsize=(10, 6))
    plt.plot(range(1, 11), inertia, marker='o')
    plt.title('Elbow Method For Optimal k')
    plt.xlabel('Number of clusters')
    plt.ylabel('Within-cluster Sum of Square')
    plt.show()
    return df_decade_scaled

scaled_spatial_data = cluster_data(spatial_data)


In [ ]:
kmeans = KMeans(n_clusters=3, random_state=42)
merged_df['SpatialCluster'] = kmeans.fit_predict(scaled_spatial_data)
print(merged_df['SpatialCluster'])
plt.figure(figsize=(10, 6))
plt.scatter(scaled_spatial_data['longitude'], scaled_spatial_data['latitude'], c=merged_df['SpatialCluster'], cmap='viridis', marker='o')
plt.title('Visualization of clustered data', fontsize=14)
plt.xlabel('Scaled X', fontsize=12)
plt.ylabel('Scaled Y', fontsize=12)
plt.show()

In [ ]:
spatial_centers = kmeans.cluster_centers_
spatial_radii = []
for i in range(3):
        cluster_points = scaled_spatial_data[merged_df['SpatialCluster'] == i]
        max_distance = max(np.linalg.norm(cluster_points - spatial_centers[i], axis=1))
        spatial_radii.append(max_distance)
        print(spatial_radii[i])

In [ ]:
text_data = merged_df[['text']]
text_data.head()

In [ ]:
text_data['totalwords'] = text_data['text'].str.count(' ') + 1
text_data.describe

In [ ]:
# Keeps the rows that totalwords > 3
text_data=text_data.loc[text_data['totalwords'] > 3]
text_data.describe